In [0]:
# Célula de Mapeamento: vincula o padrão do arquivo ao id_atleta
from pyspark.sql.functions import col

# 1. Pega o id_atleta cadastrado na dim_atleta
id_atleta_cadastrado = spark.table("workspace.default.dim_atleta").select("id_atleta").first()[0]

# 2. Cria o registro de de-para para o Rafael
data_map = [
    ("rafael", "workouts_rafael_souto_de_souza", id_atleta_cadastrado)
]

colunas_map = ["termo_busca", "padrao_arquivo", "id_atleta"]
df_map = spark.createDataFrame(data_map, colunas_map)

# 3. Salva a tabela no catálogo
df_map.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_atleta_trainingpeaks")

print("Tabela workspace.default.dim_atleta_trainingpeaks criada com sucesso!")
display(spark.table("workspace.default.dim_atleta_trainingpeaks"))

Tabela workspace.default.dim_atleta_trainingpeaks criada com sucesso!


termo_busca,padrao_arquivo,id_atleta
rafael,workouts_rafael_souto_de_souza,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e


In [0]:
print("Quantidade de atletas:")
print(spark.table("workspace.default.dim_atleta").count())

print("Estrutura da dim_atleta:")
spark.table("workspace.default.dim_atleta").printSchema()

display(
    spark.table("workspace.default.dim_atleta")
)

Quantidade de atletas:
3
Estrutura da dim_atleta:
root
 |-- id_atleta: string (nullable = true)
 |-- data_nascimento: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- altura_cm: double (nullable = true)
 |-- peso_kg: double (nullable = true)
 |-- objetivo_principal: string (nullable = true)
 |-- prova_alvo: string (nullable = true)
 |-- frequencia_treino_semanal: string (nullable = true)
 |-- horario_treino: string (nullable = true)
 |-- trainingpeaks_link: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- idade_anos: long (nullable = true)



id_atleta,data_nascimento,sexo,altura_cm,peso_kg,objetivo_principal,prova_alvo,frequencia_treino_semanal,horario_treino,trainingpeaks_link,data_ingestao,idade_anos
e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,30/01/1992,Masculino,169.0,70.0,Estreia em prova Hyrox / Treinamento Híbrido,HYROX Rio de Janeiro novembro / 2026,4 dias na semana 1h a 1h30,Varia conforme o dia,null,2026-09-13T17:10:25.755Z,34
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,06/06/1985,Masculino,170.0,72.0,Composição corporal (ganho de massa magra associado a endurance),não,"7 dias , cerca de 1 hora por dia",Varia conforme o dia,null,2026-09-13T17:10:25.755Z,41
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,22/02/1982,Masculino,173.0,82.0,Composição corporal (ganho de massa magra associado a endurance),11/10/2026,Cercs de 1:30 dividido,Manhã — em jejum,Rafael_Souto.,2026-09-13T17:10:25.755Z,44


In [0]:
# Ler a dimensão principal de atletas, gerada a partir da anamnese
df_atletas = spark.table("workspace.default.dim_atleta")

# Ler a tabela de vínculo entre arquivo do TrainingPeaks e atleta
df_de_para = spark.table("workspace.default.dim_atleta_trainingpeaks")

# Mostrar o cadastro atual do de-para
print("Tabela de-para atual:")
display(df_de_para)

# Verificar se todo id_atleta do de-para existe na dim_atleta atual
df_validacao_de_para = (
    df_de_para
    .join(
        df_atletas.select("id_atleta"),
        on="id_atleta",
        how="left_anti"
    )
)

# Se aparecer zero registros, o vínculo está correto
print(
    "IDs do de-para que NÃO existem na dim_atleta:",
    df_validacao_de_para.count()
)

display(df_validacao_de_para)

Tabela de-para atual:


termo_busca,padrao_arquivo,id_atleta
rafael,workouts_rafael_souto_de_souza,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e


IDs do de-para que NÃO existem na dim_atleta: 0


id_atleta,termo_busca,padrao_arquivo


In [0]:
from pyspark.sql.functions import (
    col, lower, round, when, expr, sha2, concat_ws, coalesce, current_timestamp
)

# 1. Definir o caminho dos arquivos CSV de treinos no Volume
caminho_volume = "/Volumes/workspace/default/treinos_trainingpeaks/*.csv"

# 2. Ler todos os CSVs de treinos presentes no Volume
# O _metadata.file_name captura o nome exato do arquivo que está sendo lido
df_bruto = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(caminho_volume)
    .withColumn("caminho_arquivo", col("_metadata.file_path"))
    .withColumn("nome_arquivo", col("_metadata.file_name"))
)

# 3. Ler a tabela de mapeamento de-para (arquivo -> atleta)
df_map = spark.table("workspace.default.dim_atleta_trainingpeaks")

# 4. Vincular o treino ao id_atleta correspondente via termo de busca no nome do arquivo
df_com_atleta = df_bruto.join(
    df_map,
    lower(df_bruto["caminho_arquivo"]).contains(lower(df_map["termo_busca"])),
    "inner"
)

# 5. Criar coluna auxiliar para duração em horas (prioriza tempo realizado, senão planejado)
df_com_duracao = df_com_atleta.withColumn(
    "duracao_base_horas",
    when(col("TimeTotalInHours").isNotNull() & (col("TimeTotalInHours") > 0), col("TimeTotalInHours"))
    .otherwise(coalesce(col("PlannedDuration"), expr("0.0")))
)

# 6. Padronização das colunas analíticas, métricas e identificador único
df_treinos_formatados = df_com_duracao.select(
    # Identificador único da sessão de treino via Hash SHA-256
    sha2(
        concat_ws(
            "_", 
            col("id_atleta"), 
            col("WorkoutDay"), 
            coalesce(col("WorkoutType"), expr("'Geral'")), 
            coalesce(col("Title"), expr("'SemTitulo'"))
        ), 
        256
    ).alias("id_treino"),
    
    col("id_atleta"),
    col("WorkoutDay").cast("string").alias("data_treino"),
    col("WorkoutType").alias("modalidade"),
    col("Title").alias("titulo_treino"),
    col("WorkoutDescription").alias("descricao_treino"),
    
    # Duração calculada em minutos
    round(col("duracao_base_horas") * 60, 1).alias("duracao_minutos"),
    
    # Distância em quilômetros (prioriza realizada, senão planejada)
    when(col("DistanceInMeters").isNotNull() & (col("DistanceInMeters") > 0), round(col("DistanceInMeters") / 1000.0, 2))
    .otherwise(round(coalesce(col("PlannedDistanceInMeters"), expr("0.0")) / 1000.0, 2)).alias("distancia_km"),
    
    col("HeartRateAverage").cast("int").alias("fc_media_bpm"),
    col("HeartRateMax").cast("int").alias("fc_max_bpm"),
    col("TSS").alias("tss"),
    
    # Regra nutricional baseada no tipo de estímulo e duração
    when((lower(col("Title")).like("%fartlek%") | lower(col("Title")).like("%tiro%")), "REG005")
    .when((lower(col("Title")).like("%rodagem%") | lower(col("Title")).like("%continuo%")) & (col("duracao_base_horas") >= 1.5), "REG002")
    .otherwise("REG001").alias("id_regra_nutricao"),
    
    # Metadados de rastreabilidade
    col("nome_arquivo").alias("nome_arquivo_origem"),
    current_timestamp().alias("data_ingestao_treino")
)

# 7. Criar uma visão temporária para o comando MERGE
df_treinos_formatados.createOrReplaceTempView("view_novos_treinos")

# 8. Garantir que a tabela Delta exista com a estrutura correta
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.fato_sessao_treino (
    id_treino STRING,
    id_atleta STRING,
    data_treino STRING,
    modalidade STRING,
    titulo_treino STRING,
    descricao_treino STRING,
    duracao_minutos DOUBLE,
    distancia_km DOUBLE,
    fc_media_bpm INT,
    fc_max_bpm INT,
    tss DOUBLE,
    id_regra_nutricao STRING,
    nome_arquivo_origem STRING,
    data_ingestao_treino TIMESTAMP
) USING DELTA;
""")

# 9. Executar o MERGE (atualiza treinos modificados e insere novos sem duplicar)
spark.sql("""
MERGE INTO workspace.default.fato_sessao_treino AS target
USING view_novos_treinos AS source
ON target.id_treino = source.id_treino
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

print("Fato de treinos atualizada com sucesso!")
print("Total de treinos na tabela:", spark.table("workspace.default.fato_sessao_treino").count())

# 10. Exibir as primeiras sessões de treino para conferência
display(
    spark.table("workspace.default.fato_sessao_treino")
    .select("id_treino", "data_treino", "modalidade", "titulo_treino", "duracao_minutos", "distancia_km", "id_regra_nutricao")
)

Fato de treinos atualizada com sucesso!
Total de treinos na tabela: 4


id_treino,data_treino,modalidade,titulo_treino,duracao_minutos,distancia_km,id_regra_nutricao
6b56bdd40e1d6a05e474542d23a905d63ba14e09771165710757cacff5fec45e,2026-09-07,Run,Rodagem + final forte,90.0,14.8,REG002
43eff98798fbb78fe16e19662b28632690819a3c289786bf38cbb42f67189bfd,2026-09-08,Run,fartlek,30.0,11.98,REG005
ab11cb42b599b5f2c2b41bbec0d50e2b55099d4d849c4cd82259a40dcd512f6d,2026-09-07,Run,Rodagem + final forte,90.0,14.8,REG002
0d6b938e8d495d44ffdf494026896c421162c40899647139869600b555961dda,2026-09-08,Run,fartlek,30.0,11.98,REG005


In [0]:
# Ler a tabela fato de treinos
df_fato_treinos = spark.table(
    "workspace.default.fato_sessao_treino"
)

# Conferir a quantidade total de treinos
print("Total de treinos:", df_fato_treinos.count())

# Conferir a quantidade de atletas com treinos importados
print(
    "Atletas com treinos:",
    df_fato_treinos.select("id_atleta").distinct().count()
)

# Conferir os principais campos dos treinos
display(
    df_fato_treinos.select(
        "id_atleta",
        "data_treino",
        "modalidade",
        "titulo_treino",
        "duracao_minutos",
        "distancia_km",
        "id_regra_nutricao"
    ).orderBy("data_treino")
)

Total de treinos: 4
Atletas com treinos: 1


id_atleta,data_treino,modalidade,titulo_treino,duracao_minutos,distancia_km,id_regra_nutricao
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-07,Run,Rodagem + final forte,90.0,14.8,REG002
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-07,Run,Rodagem + final forte,90.0,14.8,REG002
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-08,Run,fartlek,30.0,11.98,REG005
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-08,Run,fartlek,30.0,11.98,REG005


In [0]:
# CÉLULA 1: Ingestão Incremental de Treinos do TrainingPeaks (Volume -> Fato)
from pyspark.sql.functions import (
    col, input_file_name, lower, round, when, expr, sha2, concat_ws, coalesce
)

# 1. Caminho do Volume com os CSVs de treinos
caminho_volume = "/Volumes/workspace/default/treinos_trainingpeaks/*.csv"

# 2. Ler todos os CSVs presentes no volume
df_bruto = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(caminho_volume) \
    .withColumn("caminho_arquivo", col("_metadata.file_path"))

# 3. Vincular com o id_atleta da tabela dim_atleta_trainingpeaks
df_map = spark.table("workspace.default.dim_atleta_trainingpeaks")

df_com_atleta = df_bruto.join(
    df_map,
    lower(df_bruto["caminho_arquivo"]).contains(lower(df_map["termo_busca"])),
    "inner"
)

# 4. Padronização e extração de métricas
df_treinos_formatados = df_com_atleta.select(
    sha2(concat_ws("_", col("id_atleta"), col("WorkoutDay"), col("Title")), 256).alias("id_treino"),
    col("id_atleta"),
    col("WorkoutDay").cast("string").alias("data_treino"),
    col("WorkoutType").alias("modalidade"),
    col("Title").alias("titulo_treino"),
    col("WorkoutDescription").alias("descricao_treino"),
    
    # Duração em minutos (prioriza tempo realizado se houver, senão planejado)
    when(col("TimeTotalInHours").isNotNull() & (col("TimeTotalInHours") > 0), round(col("TimeTotalInHours") * 60, 1))
        .otherwise(round(coalesce(col("PlannedDuration"), expr("0.0")) * 60, 1)).alias("duracao_minutos"),
        
    # Distância em km
    when(col("DistanceInMeters").isNotNull() & (col("DistanceInMeters") > 0), round(col("DistanceInMeters") / 1000.0, 2))
        .otherwise(round(coalesce(col("PlannedDistanceInMeters"), expr("0.0")) / 1000.0, 2)).alias("distancia_km"),
        
    col("HeartRateAverage").cast("int").alias("fc_media_bpm"),
    col("HeartRateMax").cast("int").alias("fc_max_bpm"),
    col("TSS").alias("tss"),
    
    # Classificação da regra nutricional (Zonas/Intensidade x Duração)
    when((lower(col("Title")).like("%fartlek%") | lower(col("Title")).like("%tiro%")), "REG005")
    .when((lower(col("Title")).like("%rodagem%") | lower(col("Title")).like("%continuo%")) & (coalesce(col("PlannedDuration"), col("TimeTotalInHours")) >= 1.5), "REG002")
    .otherwise("REG001").alias("id_regra_nutricao")
)

# 5. Salvar na fato_sessao_treino com MERGE (upsert sem duplicar)
df_treinos_formatados.createOrReplaceTempView("view_novos_treinos")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.fato_sessao_treino (
    id_treino STRING,
    id_atleta STRING,
    data_treino STRING,
    modalidade STRING,
    titulo_treino STRING,
    descricao_treino STRING,
    duracao_minutos DOUBLE,
    distancia_km DOUBLE,
    fc_media_bpm INT,
    fc_max_bpm INT,
    tss DOUBLE,
    id_regra_nutricao STRING
) USING DELTA;
""")

spark.sql("""
MERGE INTO workspace.default.fato_sessao_treino AS target
USING view_novos_treinos AS source
ON target.id_treino = source.id_treino
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

print("Fato de treinos atualizada com sucesso!")

Fato de treinos atualizada com sucesso!
